# AIT04104 – Topic 6: Ensuring Code Quality: Testing and Documentation

**Practical Session Notebook**  
Basic Technician Certificate in Artificial Intelligence and Machine Learning  
NTA Level 4 · Academic Year 2026/2027  
*Prepared by: Mr. Joseph Magiha · Baobab Institute of Tanzania*

---

This notebook contains runnable demonstrations for every example in the Topic 6 study notes.
Exercises are **not** included here — see the exercise sheet.

**Contents**
1. [Section 6.1.2 – Code Under Test: `math_utils`](#s612)
2. [Section 6.1.2 – unittest Test Suite](#s612-unittest)
3. [Section 6.1.4 – pytest Test Suite](#s614-pytest)
4. [Section 6.2.1 – Inline Comments: Good vs Poor](#s621)
5. [Section 6.2.2 – Single-Line Docstrings](#s622-single)
6. [Section 6.2.2 – Multi-Line Docstrings (Google Style)](#s622-multi)
7. [Section 6.2.2 – Accessing Docstrings at Runtime](#s622-runtime)
8. [Section 6.2.3 – Type Hints](#s623)
9. [Section 6.2.4 – Full Module Documentation Example](#s624)

---
## Setup

Run this cell first. It installs `pytest` and `ipytest` (needed to run pytest inside a notebook).

In [12]:
import sys
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest', 'ipytest', '--quiet'])
import ipytest
ipytest.autoconfig()
print('Setup complete.')


Setup complete.


---
<a id="s612"></a>
## Section 6.1.2 – The Code Under Test: `math_utils`

The four utility functions below are the **subject** of all tests in this notebook.
We define them directly in the notebook so every subsequent cell can import them.

In [13]:
# math_utils.py  (defined here so all cells below can use the functions)

def add(a, b):
    """Return the sum of a and b."""
    return a + b


def is_even(n):
    """Return True if n is even, False otherwise."""
    return n % 2 == 0


def safe_divide(a, b):
    """Divide a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


def celsius_to_fahrenheit(c):
    """Convert Celsius temperature to Fahrenheit."""
    return c * 9 / 5 + 32


# Quick sanity check
print("add(2, 3)               =", add(2, 3))
print("is_even(10)             =", is_even(10))
print("safe_divide(10, 2)      =", safe_divide(10, 2))
print("celsius_to_fahrenheit(0)=", celsius_to_fahrenheit(0))

add(2, 3)               = 5
is_even(10)             = True
safe_divide(10, 2)      = 5.0
celsius_to_fahrenheit(0)= 32.0


---
<a id="s612-unittest"></a>
## Section 6.1.2 – unittest Test Suite

`unittest` is Python's built-in testing framework.  
Tests live in **classes** that inherit from `unittest.TestCase`.  
Each test method starts with `test_` and uses `self.assert*()` methods.

> **Running in a notebook:** we pass `argv=['']` and `exit=False` so `unittest.main()`
> works correctly inside Jupyter.

In [14]:
import unittest


class TestAdd(unittest.TestCase):

    def test_add_positive_numbers(self):
        self.assertEqual(add(2, 3), 5)

    def test_add_negative_numbers(self):
        self.assertEqual(add(-1, -1), -2)

    def test_add_with_zero(self):
        self.assertEqual(add(0, 100), 100)

    def test_add_floats(self):
        # assertAlmostEqual is used for floats to avoid precision errors
        self.assertAlmostEqual(add(0.1, 0.2), 0.3, places=5)


class TestIsEven(unittest.TestCase):

    def test_even_number_returns_true(self):
        self.assertTrue(is_even(10))

    def test_odd_number_returns_false(self):
        self.assertFalse(is_even(7))

    def test_zero_is_even(self):
        self.assertTrue(is_even(0))

    def test_negative_even(self):
        self.assertTrue(is_even(-4))


class TestSafeDivide(unittest.TestCase):

    def test_normal_division(self):
        self.assertEqual(safe_divide(10, 2), 5.0)

    def test_division_by_zero_raises_value_error(self):
        # assertRaises checks that the expected exception IS raised
        with self.assertRaises(ValueError):
            safe_divide(10, 0)


class TestCelsiusToFahrenheit(unittest.TestCase):

    def test_freezing_point(self):
        self.assertEqual(celsius_to_fahrenheit(0), 32)

    def test_boiling_point(self):
        self.assertEqual(celsius_to_fahrenheit(100), 212)

    def test_body_temperature(self):
        self.assertAlmostEqual(celsius_to_fahrenheit(37), 98.0, places=1)


# Run the test suite
unittest.main(argv=[''], exit=False, verbosity=2)

test_add_floats (__main__.TestAdd.test_add_floats) ... ok
test_add_negative_numbers (__main__.TestAdd.test_add_negative_numbers) ... ok
test_add_positive_numbers (__main__.TestAdd.test_add_positive_numbers) ... ok
test_add_with_zero (__main__.TestAdd.test_add_with_zero) ... ok
test_body_temperature (__main__.TestCelsiusToFahrenheit.test_body_temperature) ... FAIL
test_boiling_point (__main__.TestCelsiusToFahrenheit.test_boiling_point) ... ok
test_freezing_point (__main__.TestCelsiusToFahrenheit.test_freezing_point) ... ok
test_even_number_returns_true (__main__.TestIsEven.test_even_number_returns_true) ... ok
test_negative_even (__main__.TestIsEven.test_negative_even) ... ok
test_odd_number_returns_false (__main__.TestIsEven.test_odd_number_returns_false) ... ok
test_zero_is_even (__main__.TestIsEven.test_zero_is_even) ... ok
test_division_by_zero_raises_value_error (__main__.TestSafeDivide.test_division_by_zero_raises_value_error) ... ok
test_normal_division (__main__.TestSafeDivide.t

**Reading the output**

- Each line shows `<TestClass>.<test_method> ... ok` when the test passes.
- A final summary line reports `Ran N tests in Xs` followed by `OK`.
- If a test fails you see `FAIL` and a diff explaining what went wrong.

---
<a id="s614-pytest"></a>
## Section 6.1.4 – pytest Test Suite

`pytest` does **not** require test classes.  
Plain functions whose names start with `test_` are collected automatically.  
Assertions use Python's built-in `assert` — no special method names needed.

The `%%ipytest` magic at the top of a cell collects and runs all `test_*` functions in that cell.

In [15]:
%%ipytest -v

import pytest

# --- Tests for add() ---
def test_add_positive_numbers():
    assert add(2, 3) == 5

def test_add_negative_numbers():
    assert add(-1, -1) == -2

def test_add_floats():
    assert abs(add(0.1, 0.2) - 0.3) < 1e-5

# --- Tests for is_even() ---
def test_even_number_returns_true():
    assert is_even(10) is True

def test_odd_number_returns_false():
    assert is_even(7) is False

# --- Tests for safe_divide() ---
def test_normal_division():
    assert safe_divide(10, 2) == 5.0

def test_division_by_zero_raises_value_error():
    with pytest.raises(ValueError):
        safe_divide(10, 0)

# --- Parametrised test: run the same test with multiple inputs ---
@pytest.mark.parametrize("celsius, expected_f", [
    (0,    32.0),
    (100,  212.0),
    (-40,  -40.0),   # The one temperature where Celsius equals Fahrenheit
    (37,   98.6),
])
def test_celsius_to_fahrenheit(celsius, expected_f):
    assert abs(celsius_to_fahrenheit(celsius) - expected_f) < 0.1

======================================= test session starts =======================================
platform win32 -- Python 3.11.9, pytest-9.0.3, pluggy-1.6.0
rootdir: c:\Users\Administrator\Desktop\AI AND ML\python-exe
collected 11 items

t_bdb2e341ae314854b6ff69cbd3eff903.py ...........                                            [100%]

======================================= 11 passed in 0.60s ========================================


**Key advantage — `@pytest.mark.parametrize`**  
The decorator runs the same test function with multiple input/expected pairs.  
Each combination appears as a separate line in the output (e.g. `test_celsius_to_fahrenheit[0-32.0]`),
so you know exactly which case failed.

---
<a id="s621"></a>
## Section 6.2.1 – Inline Comments: Good vs Poor

Comments start with `#`. They are **ignored by Python** — they exist only for human readers.  
The golden rule: **explain *why*, not *what*.**

In [16]:
# ── Poor comment: merely restates what the code already says ──────────────
x = 10
x = x + 1   # add 1 to x    <-- states the obvious

# ── Good comment: explains the non-obvious reason ──────────────────────────
x = 10
x = x + 1   # offset by 1 because the dataset uses 1-based indexing

print('x =', x)

x = 11


In [17]:
# ── Good comment: explains a deliberate algorithmic choice ─────────────────
# Using insertion sort here instead of quicksort because the input list
# is expected to be nearly sorted (< 5 items out of place), and insertion
# sort performs at O(n) in that case.
def sort_predictions(preds):
    return sorted(preds)   # stdlib sorted uses Timsort (similar idea)

print(sort_predictions([3, 1, 2, 4, 5]))

[1, 2, 3, 4, 5]



In [18]:
# ── Good comment: marks a known issue with context (TODO) ──────────────────
large_list      = list(range(100))
another_large_list = list(range(50, 150))

# TODO: replace this O(n^2) search with a hash map when dataset > 10 000 rows
found = [item for item in large_list if item in another_large_list]
print('Overlap count:', len(found))

Overlap count: 50


---
<a id="s622-single"></a>
## Section 6.2.2 – Single-Line Docstrings

A **docstring** is a string literal placed as the **first statement** inside a function, class, or module.
Unlike a comment, Python stores it on the object's `__doc__` attribute.

For simple functions, one line is enough.

In [19]:
def square(n):
    """Return the square of n."""
    return n * n


def is_palindrome(text):
    """Return True if text reads the same forwards and backwards."""
    return text == text[::-1]


print(square(5))
print(is_palindrome("racecar"))
print(is_palindrome("hello"))

25
True
False


---
<a id="s622-multi"></a>
## Section 6.2.2 – Multi-Line Docstrings (Google Style)

Functions with parameters, return values, raised exceptions, or non-trivial behaviour
deserve a **multi-line docstring** following a consistent convention.
The Google style format uses sections: `Args:`, `Returns:`, `Raises:`, `Examples:`.

In [20]:
def classify_weather(temp, humidity=None):
    """
    Classify weather conditions based on temperature and optional humidity.

    Args:
        temp (float): Temperature in degrees Celsius.
        humidity (float, optional): Relative humidity as a percentage (0-100).
            If not provided, humidity is not considered in the classification.

    Returns:
        str: A weather category label. One of: "Hot", "Warm", "Cool", "Cold".

    Raises:
        ValueError: If temp is not a number, or if humidity is outside 0-100.

    Examples:
        >>> classify_weather(35)
        "Hot"
        >>> classify_weather(22, humidity=80)
        "Warm"
    """
    if not isinstance(temp, (int, float)):
        raise ValueError(f"temp must be numeric, got {type(temp).__name__}")
    if humidity is not None and not (0 <= humidity <= 100):
        raise ValueError(f"humidity must be 0-100, got {humidity}")

    if temp > 30:
        return "Hot"
    elif temp > 20:
        return "Warm"
    elif temp > 10:
        return "Cool"
    else:
        return "Cold"


# Demo
print(classify_weather(35))
print(classify_weather(22, humidity=80))
print(classify_weather(-5))

Hot
Warm
Cold


---
<a id="s622-runtime"></a>
## Section 6.2.2 – Accessing Docstrings at Runtime

Because Python stores docstrings on the `__doc__` attribute, you can read them
programmatically — and tools like `help()` and IDEs use exactly this mechanism.

In [21]:
# help() formats the docstring in a readable way
help(classify_weather)

Help on function classify_weather in module __main__:

classify_weather(temp, humidity=None)
    Classify weather conditions based on temperature and optional humidity.
    
    Args:
        temp (float): Temperature in degrees Celsius.
        humidity (float, optional): Relative humidity as a percentage (0-100).
            If not provided, humidity is not considered in the classification.
    
    Returns:
        str: A weather category label. One of: "Hot", "Warm", "Cool", "Cold".
    
    Raises:
        ValueError: If temp is not a number, or if humidity is outside 0-100.
    
    Examples:
        >>> classify_weather(35)
        "Hot"
        >>> classify_weather(22, humidity=80)
        "Warm"



In [22]:
# __doc__ gives the raw docstring string
print(classify_weather.__doc__)


    Classify weather conditions based on temperature and optional humidity.

    Args:
        temp (float): Temperature in degrees Celsius.
        humidity (float, optional): Relative humidity as a percentage (0-100).
            If not provided, humidity is not considered in the classification.

    Returns:
        str: A weather category label. One of: "Hot", "Warm", "Cool", "Cold".

    Raises:
        ValueError: If temp is not a number, or if humidity is outside 0-100.

    Examples:
        >>> classify_weather(35)
        "Hot"
        >>> classify_weather(22, humidity=80)
        "Warm"
    


In [23]:
# In Jupyter/IPython you can also type:  classify_weather?
# (uncomment the line below to try it)
# classify_weather?

# Confirm that square() also has a stored docstring
print(square.__doc__)
print(is_palindrome.__doc__)

Return the square of n.
Return True if text reads the same forwards and backwards.


---
<a id="s623"></a>
## Section 6.2.3 – Type Hints

Type hints (Python 3.5+) annotate parameters and return values with their expected types.
Python does **not** enforce them at runtime — they are documentation for humans and static
analysis tools (e.g. `mypy`, IDE auto-complete).

In [24]:
from typing import Optional, List, Dict

# Without type hints: ambiguous — what is 'data'? what does it return?
def process_plain(data, threshold):
    ...

# With type hints: immediately clear
def process(data: List[float], threshold: float) -> List[float]:
    """Return elements of data that exceed threshold."""
    return [x for x in data if x > threshold]

print(process([1.0, 3.5, 0.8, 4.2, 2.1], 2.0))

[3.5, 4.2, 2.1]


In [25]:
# Optional[str] means the parameter can be str OR None
def greet(name: Optional[str] = None) -> str:
    if name is None:
        return "Hello, stranger!"
    return f"Hello, {name}!"

print(greet())
print(greet("Amina"))

Hello, stranger!
Hello, Amina!


In [26]:
# Dict return type
def get_stats(scores: List[float]) -> Dict[str, float]:
    return {
        "mean": sum(scores) / len(scores),
        "min":  min(scores),
        "max":  max(scores),
    }

scores = [55.0, 72.0, 88.0, 41.0, 66.0]
stats = get_stats(scores)
for k, v in stats.items():
    print(f'{k}: {v}')

mean: 64.4
min: 41.0
max: 88.0


> **Python 3.9+ shorthand:** write `list[float]` instead of `List[float]`.  
> **Python 3.10+ shorthand:** write `str | None` instead of `Optional[str]`.

---
<a id="s624"></a>
## Section 6.2.4 – Full Module Documentation Example

This cell shows a complete, production-quality `math_utils` module:
a **module-level docstring**, full **Google-style function docstrings**, and **type hints**.
Running `help(math_utils_demo)` below demonstrates what `pydoc` would generate.

### pydoc commands (run in a terminal, not in this notebook)
```bash
# Display documentation in the terminal
python -m pydoc math_utils

# Generate an HTML file (math_utils.html) in the current directory
python -m pydoc -w math_utils

# Start a local documentation web server on port 8080
python -m pydoc -p 8080
# Then open http://localhost:8080 in a browser
```

In [27]:
# We define the module as a class-level namespace so help() works inside Jupyter.
# In a real project these would be top-level definitions in math_utils.py.

class math_utils_demo:  # acts as a stand-in for the module
    """
    math_utils — Basic mathematical utility functions.

    This module provides helper functions for common arithmetic operations
    and temperature conversions, with input validation.

    Typical usage:
        from math_utils import add, celsius_to_fahrenheit
        total = add(10, 20)
        temp_f = celsius_to_fahrenheit(37)
    """

    @staticmethod
    def add(a: float, b: float) -> float:
        """
        Return the arithmetic sum of a and b.

        Args:
            a (float): First operand.
            b (float): Second operand.

        Returns:
            float: The sum a + b.

        Examples:
            >>> add(2, 3)
            5
            >>> add(-1.5, 1.5)
            0.0
        """
        return a + b

    @staticmethod
    def is_even(n: int) -> bool:
        """
        Return True if n is even, False otherwise.

        Args:
            n (int): The integer to test.

        Returns:
            bool: True when n is divisible by 2.

        Examples:
            >>> is_even(4)
            True
            >>> is_even(7)
            False
        """
        return n % 2 == 0

    @staticmethod
    def safe_divide(a: float, b: float) -> float:
        """
        Divide a by b.

        Args:
            a (float): Numerator.
            b (float): Denominator. Must not be zero.

        Returns:
            float: The quotient a / b.

        Raises:
            ValueError: If b is zero.

        Examples:
            >>> safe_divide(10, 2)
            5.0
        """
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

    @staticmethod
    def celsius_to_fahrenheit(c: float) -> float:
        """
        Convert a temperature from Celsius to Fahrenheit.

        Args:
            c (float): Temperature in degrees Celsius.

        Returns:
            float: Equivalent temperature in degrees Fahrenheit.

        Examples:
            >>> celsius_to_fahrenheit(0)
            32.0
            >>> celsius_to_fahrenheit(100)
            212.0
        """
        return c * 9 / 5 + 32


# Inspect the generated documentation
help(math_utils_demo)

Help on class math_utils_demo in module __main__:

class math_utils_demo(builtins.object)
 |  math_utils — Basic mathematical utility functions.
 |  
 |  This module provides helper functions for common arithmetic operations
 |  and temperature conversions, with input validation.
 |  
 |  Typical usage:
 |      from math_utils import add, celsius_to_fahrenheit
 |      total = add(10, 20)
 |      temp_f = celsius_to_fahrenheit(37)
 |  
 |  Static methods defined here:
 |  
 |  add(a: float, b: float) -> float
 |      Return the arithmetic sum of a and b.
 |      
 |      Args:
 |          a (float): First operand.
 |          b (float): Second operand.
 |      
 |      Returns:
 |          float: The sum a + b.
 |      
 |      Examples:
 |          >>> add(2, 3)
 |          5
 |          >>> add(-1.5, 1.5)
 |          0.0
 |  
 |  celsius_to_fahrenheit(c: float) -> float
 |      Convert a temperature from Celsius to Fahrenheit.
 |      
 |      Args:
 |          c (float): Temperature 

---
## Summary

| Section | Concept | Key takeaway |
|---------|---------|-------------|
| 6.1.2 | `unittest` | Test classes inherit `unittest.TestCase`; use `self.assert*()` methods |
| 6.1.4 | `pytest` | Plain `test_*` functions; plain `assert`; `@parametrize` for multiple inputs |
| 6.2.1 | Inline comments | Explain **why**, not what; keep them up to date |
| 6.2.2 | Docstrings | Stored on `__doc__`; Google style: `Args/Returns/Raises/Examples` |
| 6.2.3 | Type hints | Documentation for humans and tools; not enforced at runtime |
| 6.2.4 | pydoc | Reads docstrings and generates terminal or HTML documentation automatically |